In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

先读数据

In [2]:
train = pd.read_csv("../data/train.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
baseline_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
target = "Survived"

特征工程：构造新特征

In [4]:
df = train.copy()

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["CabinKnown"] = df["Cabin"].notnull().astype(int)

df[["Name", "FamilySize", "IsAlone", "CabinKnown"]].head()

,Name,FamilySize,IsAlone,CabinKnown
0,"Braund, Mr. Owen Harris",2,0,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",2,0,1
2,"Heikkinen, Miss. Laina",1,1,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",2,0,1
4,"Allen, Mr. William Henry",1,1,0


In [5]:
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [6]:
title_mapping = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
}

df["Title"] = df["Title"].replace(title_mapping)

common_titles = ["Mr", "Miss", "Mrs", "Master"]
df["Title"] = df["Title"].apply(lambda x: x if x in common_titles else "Rare")

df["Title"].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

最后合并得到剩下的全部特征

In [8]:
engineered_features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "FamilySize", "IsAlone", "CabinKnown", "Title"
]

In [9]:
def prepare_features(data, feature_list, target_col="Survived"):
    temp = data[feature_list + [target_col]].copy()
    
    if "Age" in temp.columns:
        temp["Age"] = temp["Age"].fillna(temp["Age"].median())
    if "Fare" in temp.columns:
        temp["Fare"] = temp["Fare"].fillna(temp["Fare"].median())
    if "Embarked" in temp.columns:
        temp["Embarked"] = temp["Embarked"].fillna(temp["Embarked"].mode()[0])
    
    categorical_cols = []
    for col in ["Sex", "Embarked", "Title"]:
        if col in temp.columns:
            categorical_cols.append(col)
    
    temp = pd.get_dummies(temp, columns=categorical_cols, drop_first=True)
    
    X = temp.drop(target_col, axis=1)
    y = temp[target_col]
    return X, y

准备两套数据：baseline和engineered

In [10]:
X_base, y_base = prepare_features(df, baseline_features, target)
X_eng, y_eng = prepare_features(df, engineered_features, target)

print("Baseline X shape:", X_base.shape)
print("Engineered X shape:", X_eng.shape)

Baseline X shape: (891, 8)
Engineered X shape: (891, 15)


In [12]:
#baseline
X_train_b, X_valid_b, y_train_b, y_valid_b = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)

model_base = LogisticRegression(max_iter=1000)
model_base.fit(X_train_b, y_train_b)

pred_base = model_base.predict(X_valid_b)
acc_base = accuracy_score(y_valid_b, pred_base)

print("Baseline accuracy:", acc_base)

#engineered
X_train_e, X_valid_e, y_train_e, y_valid_e = train_test_split(
    X_eng, y_eng, test_size=0.2, random_state=42, stratify=y_eng
)

model_eng = LogisticRegression(max_iter=1000)
model_eng.fit(X_train_e, y_train_e)

pred_eng = model_eng.predict(X_valid_e)
acc_eng = accuracy_score(y_valid_e, pred_eng)

print("Engineered accuracy:", acc_eng)

Baseline accuracy: 0.8044692737430168
Engineered accuracy: 0.8100558659217877


In [13]:
compare_df = pd.DataFrame({
    "version": ["baseline", "engineered"],
    "num_features": [X_base.shape[1], X_eng.shape[1]],
    "accuracy": [acc_base, acc_eng]
})

compare_df

,version,num_features,accuracy
0,baseline,8,0.804469
1,engineered,15,0.810056


In [14]:
compare_df.to_csv("../outputs/tables/feature_compare.csv", index=False)

## Feature Engineering Summary

- I added FamilySize, IsAlone, CabinKnown, and Title.
- Family-related features try to capture whether a passenger traveled alone or with others.
- Title may contain information about age, gender, and social status.
- CabinKnown uses the existence of cabin information instead of the raw Cabin column.
- The engineered feature set was compared directly with the baseline model using the same validation setup.